In [ ]:
import sys
import pathlib
import importlib 
import json
import os

import anywidget
import traitlets

# -----------------------------------------------------------------------------
# 1) Paths and static assets
# -----------------------------------------------------------------------------
ROOT_PATH = os.getcwd().split("notebooks")[0]
CSS_PATH = os.path.join(ROOT_PATH, "css", "lectures.css")
widget_css = pathlib.Path(CSS_PATH).read_text()

# -----------------------------------------------------------------------------
# 2) Import engine for get JavaScript ESM6 modules
# -----------------------------------------------------------------------------
# 1. Path logic 
root = str(pathlib.Path.cwd().parent.absolute())
if root not in sys.path: sys.path.append(root)
# 2. Imports
import vis_engine
importlib.reload(vis_engine) # For your debugging 
get_project_modules = vis_engine.get_project_modules
MODULE_ALIASES = vis_engine.MODULE_ALIASES
# -----------------------------------------------------------------------------
# 3) Module resolution for browser-safe Data URI imports
# -----------------------------------------------------------------------------
# Map project aliases to source files.
MODULE_ALIASES["dijkstra"] = "04_networks/traversal/dijkstra.js"
MODULE_ALIASES["traversalAlgorithms"] = "04_networks/traversal/traversalAlgorithms.js"
MODULE_ALIASES["networkUtils"] = "04_networks/core/networkUtils.js"

proj_modules = get_project_modules(MODULE_ALIASES, ROOT_PATH)

js_logic = """
export async function render({ model, el }) {
    // set background color for the widget
    el.style.backgroundColor = 'transparent'
    
    // import the module for drawing the force-directed layout
    const moduleUri = model.get('dijkstra_uri')
    const { drawAll } = await import(moduleUri);
    
    // create div for canvas and draw
    const divElement = document.createElement("div")
    
    // set class and id for styling and selection
    divElement.setAttribute('class', 'canvas grid-container-1-column')
    divElement.setAttribute('id', 'dijkstra-traversal')
    divElement.style.textAlign = 'center'

    // append to anywidget element
    el.appendChild(divElement);

    const test01 = model.get('network_data_01')
    const test02 = model.get('network_data_02')
    drawAll('#dijkstra-traversal', test01, test02)
}
"""

# -----------------------------------------------------------------------------
# 4) Data + widget model
# -----------------------------------------------------------------------------
network_path_01 = pathlib.Path(ROOT_PATH) / "data" / "test01.json"
network_path_02 = pathlib.Path(ROOT_PATH) / "data" / "test02.json"

with open(network_path_01, "r") as f:
    initial_data_01 = json.load(f)

with open(network_path_02, "r") as f:
    initial_data_02 = json.load(f)

class MyClassicD3Widget(anywidget.AnyWidget):
    network_data_01 = traitlets.Dict(initial_data_01).tag(sync=True)
    network_data_02 = traitlets.Dict(initial_data_02).tag(sync=True)
    dijkstra_uri = traitlets.Unicode(
        proj_modules["dijkstra"]
    ).tag(sync=True)
    _css = CSS_PATH
    _esm = js_logic

MyClassicD3Widget()